<a href="https://colab.research.google.com/github/Tahir-MD/FlyRank-Week-01/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install duckdb huggingface-hub pandas -q

import duckdb
import pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()

con.execute(f"CREATE SECRET hf_token (TYPE huggingface, TOKEN '{HF_TOKEN}')")
print("✅ Connected to Hugging Face successfully!")

✅ Connected to Hugging Face successfully!


## 1. Unit of Analysis + Time Window

**One row = one page per day.** Each row represents daily performance for a content page. I'll aggregate this to monthly level for my analysis.

**Key columns:**
- `content_hash_id` = Page identifier
- `report_date` = Date of the record
- `month` = Month of the record

**Time window:** I'm using **March 2026** as my development month. June 2026 is reserved as the final test month.

In [12]:
columns = con.execute("""
DESCRIBE SELECT *
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

print("Columns in the data : ")
print(columns['column_name'].tolist())

march_data = con.execute("""
SELECT
    content_hash_id,
    month,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    ga4_users,
    scroll_events
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

print(f"\nMarch 2026 has {len(march_data)} rows")
print(f"Unique pages : {march_data['content_hash_id'].nunique()}")
print(f"Date range : {march_data['report_date'].min()} to {march_data['report_date'].max()}")

print("\nSample data:")
print(march_data.head())

Columns in the data : 
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


March 2026 has 9841378 rows
Unique pages : 331437
Date range : 2026-03-01 00:00:00 to 2026-03-31 00:00:00

Sample data:
            content_hash_id    month report_date  gsc_impressions  gsc_clicks  \
0  content_b7e512995f79d5a6  2026-03  2026-03-01               20           0   
1  content_05597932fe4da067  2026-03  2026-03-01                1           0   
2  content_7a105f548d9c6916  2026-03  2026-03-01              125           1   
3  content_905aa32a0230694e  2026-03  2026-03-01                7           0   
4  content_a3ea9792f793ec72  2026-03  2026-03-01               11           0   

   gsc_avg_position  ga4_sessions  ga4_users  scroll_events  
0          3.350000          <NA>       <NA>           <NA>  
1          0.000000          <NA>       <NA>           <NA>  
2          4.928000          <NA>       <NA>           <NA>  
3          4.000000          <NA>       <NA>           <NA>  
4          2.272727          <NA>       <NA>           <NA>  


# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tahir-MD/FlyRank-Week-01/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 2. Fields: Feature / Label / Context / Excluded

### FEATURES (What I'll use to predict)
| Field | Description |
|-------|-------------|
| `gsc_impressions` | Number of times page appeared in search |
| `gsc_clicks` | Number of clicks from search |
| `gsc_avg_position` | Average search ranking position |
| `ga4_sessions` | Number of user sessions |
| `ga4_users` | Number of unique users |
| `scroll_events` | User engagement signal |
| `ctr` | Click-through rate (calculated: clicks/impressions) |

### LABEL (What I'm predicting)
| Field | Description |
|-------|-------------|
| `is_declining` | True if page is losing traffic (proxy label I create) |

### CONTEXT (For identification only)
| Field | Purpose |
|-------|---------|
| `content_hash_id` | Page identifier |
| `month` | Time period |

### EXCLUDED (Intentionally left out)
| Field | Why |
|-------|-----|
| `client_hash_id` | Avoid memorizing client patterns |
| `client_has_gsc` / `client_has_ga4` | Not relevant to page performance |
| Future data | Would cause data leakage |

In [13]:
available = columns['column_name'].tolist()

features = ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_users', 'scroll_events']
context = ['content_hash_id', 'month', 'report_date']

print("Available features:")
for f in features:
    if f in available:
        print(f"  ✓ {f}")
    else:
        print(f"  ✗ {f} (missing)")

print(f"\nContext fields: {[f for f in context if f in available]}")

print("\nNote: CTR will be calculated as gsc_clicks / gsc_impressions")

sample_data = con.execute("""
SELECT
    content_hash_id,
    month,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    scroll_events
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
LIMIT 5
""").df()

print("\nSample with our fields:")
print(sample_data)

Available features:
  ✓ gsc_impressions
  ✓ gsc_clicks
  ✓ gsc_avg_position
  ✓ ga4_sessions
  ✓ ga4_users
  ✓ scroll_events

Context fields: ['content_hash_id', 'month', 'report_date']

Note: CTR will be calculated as gsc_clicks / gsc_impressions

Sample with our fields:
            content_hash_id    month report_date  gsc_impressions  gsc_clicks  \
0  content_b7e512995f79d5a6  2026-03  2026-03-01               20           0   
1  content_05597932fe4da067  2026-03  2026-03-01                1           0   
2  content_7a105f548d9c6916  2026-03  2026-03-01              125           1   
3  content_905aa32a0230694e  2026-03  2026-03-01                7           0   
4  content_a3ea9792f793ec72  2026-03  2026-03-01               11           0   

   gsc_avg_position  ga4_sessions  scroll_events  
0          3.350000          <NA>           <NA>  
1          0.000000          <NA>           <NA>  
2          4.928000          <NA>           <NA>  
3          4.000000          <NA>   

## 3. Verify it with queries (grain, counts, missing values, windows)

Three queries to verify my data contract:
1. **Grain:** Check that each page has one row per day
2. **Counts:** How many pages and what's the date range
3. **Availability:** How much data is complete and usable

In [14]:
print("\n1️. GRAIN CHECK:")

grain_check = con.execute("""
SELECT
    month,
    COUNT(DISTINCT content_hash_id) as unique_pages,
    COUNT(*) as total_rows,
    COUNT(DISTINCT report_date) as days
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY month
""").df()

print(grain_check)

if len(grain_check) > 0:
    pages = grain_check['unique_pages'].iloc[0]
    days = grain_check['days'].iloc[0]
    rows = grain_check['total_rows'].iloc[0]
    expected = pages * days
    if rows == expected:
        print(f"Perfect grain: {pages} pages × {days} days = {rows} rows")
    else:
        print(f"Some days missing: {pages} pages × {days} days = {expected} expected, got {rows}")

print("\n2️. DATE RANGE & COUNTS:")
date_counts = con.execute("""
SELECT
    month,
    COUNT(DISTINCT content_hash_id) as page_count,
    MIN(report_date) as first_date,
    MAX(report_date) as last_date
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
WHERE month >= '2026-01-01' AND month <= '2026-06-01'
GROUP BY month
ORDER BY month
""").df()

print(date_counts)
print(f"\nDate range: {date_counts['first_date'].min()} to {date_counts['last_date'].max()}")

print("\n3️. DATA AVAILABILITY:")
availability = con.execute("""
SELECT
    month,
    COUNT(*) as total_rows,
    COUNT(*) FILTER (WHERE gsc_impressions > 0) as has_impressions,
    COUNT(*) FILTER (WHERE gsc_clicks > 0) as has_clicks,
    COUNT(*) FILTER (WHERE gsc_avg_position IS NOT NULL) as has_position,
    COUNT(*) FILTER (WHERE ga4_sessions > 0) as has_sessions
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY month
""").df()

print(availability)

if len(availability) > 0:
    row = availability.iloc[0]
    print(f"\nMarch 2026 completeness:")
    print(f"  Impressions: {row['has_impressions']}/{row['total_rows']} ({row['has_impressions']/row['total_rows']*100:.1f}%)")
    print(f"  Clicks: {row['has_clicks']}/{row['total_rows']} ({row['has_clicks']/row['total_rows']*100:.1f}%)")
    print(f"  Position: {row['has_position']}/{row['total_rows']} ({row['has_position']/row['total_rows']*100:.1f}%)")
    print(f"  Sessions: {row['has_sessions']}/{row['total_rows']} ({row['has_sessions']/row['total_rows']*100:.1f}%)")


1️. GRAIN CHECK:
     month  unique_pages  total_rows  days
0  2026-03        331437     9841378    31
Some days missing: 331437 pages × 31 days = 10274547 expected, got 9841378

2️. DATE RANGE & COUNTS:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

     month  page_count first_date  last_date
0  2026-02      321546 2026-02-01 2026-02-28
1  2026-03      331437 2026-03-01 2026-03-31
2  2026-04      362172 2026-04-01 2026-04-30
3  2026-05      389153 2026-05-01 2026-05-31
4  2026-06      409205 2026-06-01 2026-06-30

Date range: 2026-02-01 00:00:00 to 2026-06-30 00:00:00

3️. DATA AVAILABILITY:
     month  total_rows  has_impressions  has_clicks  has_position  \
0  2026-03     9841378          3611061      417981       3611061   

   has_sessions  
0        410335  

March 2026 completeness:
  Impressions: 3611061/9841378 (36.7%)
  Clicks: 417981/9841378 (4.2%)
  Position: 3611061/9841378 (36.7%)
  Sessions: 410335/9841378 (4.2%)


## 4. Data limits

I'm aggregating daily data to monthly level for my content refresh lane.

### Features I create:
- Monthly totals for impressions, clicks, sessions
- Monthly averages for position
- Calculated CTR (clicks / impressions)

### How I create the label:
- `is_declining = True` if impressions dropped more than 20% from previous month
- This is a **proxy label** - it captures the trend I want to predict

In [15]:
monthly_data = con.execute("""
SELECT
    content_hash_id,
    DATE_TRUNC('month', report_date) as month,
    SUM(gsc_impressions) as impressions,
    SUM(gsc_clicks) as clicks,
    AVG(gsc_avg_position) as avg_position,
    SUM(ga4_sessions) as sessions,
    SUM(ga4_users) as users,
    SUM(scroll_events) as scrolls,
    COUNT(DISTINCT report_date) as days_active,
    -- Calculate CTR (clicks / impressions)
    SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) as ctr
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
WHERE report_date >= '2026-01-01' AND report_date <= '2026-05-01'
GROUP BY content_hash_id, DATE_TRUNC('month', report_date)
""").df()

monthly_data['month_str'] = monthly_data['month'].dt.strftime('%Y-%m')

print(f"Created monthly data: {len(monthly_data)} rows")
print(f"Months: {sorted(monthly_data['month_str'].unique())}")
print(f"\nFields created:")
print(monthly_data.columns.tolist())

print("\nSample:")
print(monthly_data.head())

print("Adding Proxy Label: is_declining")

monthly_data = monthly_data.sort_values(['content_hash_id', 'month'])

monthly_data['prev_impressions'] = monthly_data.groupby('content_hash_id')['impressions'].shift(1)

monthly_data['is_declining'] = 0
mask = monthly_data['prev_impressions'].notna() & (monthly_data['prev_impressions'] > 0)
monthly_data.loc[mask, 'is_declining'] = (
    monthly_data.loc[mask, 'impressions'] / monthly_data.loc[mask, 'prev_impressions'] < 0.8
).astype(int)

print(f"Added is_declining label")
print(f"Declining pages: {monthly_data['is_declining'].sum():,} out of {len(monthly_data):,}")
print(f"Overall decline rate: {monthly_data['is_declining'].mean()*100:.1f}%")

march = monthly_data[monthly_data['month_str'] == '2026-03']
print(f"\nMarch 2026: {len(march)} pages")
print(f"Declining in March: {march['is_declining'].sum()} ({march['is_declining'].mean()*100:.1f}%)")

print("\nComparison: Declining vs Stable Pages (March 2026)")
comparison = march.groupby('is_declining').agg({
    'impressions': 'mean',
    'clicks': 'mean',
    'avg_position': 'mean',
    'ctr': 'mean'
}).round(2)
print(comparison)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Created monthly data: 1640774 rows
Months: ['2026-01', '2026-02', '2026-03', '2026-04', '2026-05']

Fields created:
['content_hash_id', 'month', 'impressions', 'clicks', 'avg_position', 'sessions', 'users', 'scrolls', 'days_active', 'ctr', 'month_str']

Sample:
            content_hash_id      month  impressions  clicks  avg_position  \
0  content_cc02e6a397d2ffc2 2026-01-01       2594.0     8.0      2.301850   
1  content_b2c8677f822e152a 2026-01-01       4603.0    14.0      5.326586   
2  content_c6ac71d72dbccc86 2026-01-01        315.0     1.0      7.281478   
3  content_7916e8f527f1c672 2026-01-01        246.0     0.0     27.480229   
4  content_c4a9f203b6708ddf 2026-01-01       2031.0     2.0      5.288123   

   sessions  users  scrolls  days_active       ctr month_str  
0       5.0    5.0      1.0           31  0.003084   2026-01  
1      12.0   11.0      3.0           31  0.003041   2026-01  
2       0.0    0.0      0.0           31  0.003175   2026-01  
3       1.0    1.0     

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.